# 12 — Diagnóstico del MBE (prueba decisiva H1 vs H5)

Resuelve la contradicción abierta del entregable 3: si la partición fuera intercambiable,
el MBE debería ser ≈ 0, pero se observó **−10,41** en el clúster 6.

- **H1**: el shuffle no fue efectivo en la corrida original → `mean(y_test)` difiere de
  `mean(y_train)` aproximadamente en la magnitud del MBE.
- **H5**: el sesgo vive en la predicción/evaluación (alineación X/y) → los splits son
  intercambiables pero `mean(pred)` queda desplazada.

El script entrena los modelos por clúster, imprime medias por split, `mean(pred)`, MBE y
un veredicto por clúster, y guarda `results/diagnostico_mbe.json`.

**Configuración del notebook:** acelerador **GPU T4 x2** (`NvidiaTeslaT4`) e Internet activado,
con el dataset `lucasiturriago/pruebassaber11` adjunto en las fuentes de datos.

In [ ]:
!git clone https://github.com/liturriago/EquidadAlgoritmica-SaberXAI.git
%cd EquidadAlgoritmica-SaberXAI
!pip install -e . -q

In [ ]:
# Resolver el dataset: adjunto en /kaggle/input o descarga con kagglehub (fallback)
from pathlib import Path

cands = [
    Path('/kaggle/input/pruebassaber11'),
    Path('/kaggle/input/datasets/lucasiturriago/pruebassaber11'),
]
data_dir = next((p for p in cands if (p / 'clase_0.parquet').exists()), None)
if data_dir is None:
    import kagglehub
    data_dir = Path(kagglehub.dataset_download('lucasiturriago/pruebassaber11'))
print('data_dir =', data_dir)

# Helper: ejecuta un comando y aborta el notebook si falla (los !python no abortan)
import shlex
import subprocess

def run(cmd):
    print(">>", cmd)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f"Falló (exit {r.returncode}): {cmd}"


In [ ]:
# Entrena los modelos por clúster (XGBoost + MLP, 0-6) y sus métricas por clúster.
# Si esta sesión ya tiene models/xgb_cluster_*.json, se puede comentar esta celda.
run(f"python scripts/train.py --config configs/lca_clusters.yaml --mode cluster --data-dir {shlex.quote(str(data_dir))}")

In [ ]:
# Prueba decisiva: medias por split vs mean(pred) por clúster
run(f"python scripts/diagnostico_mbe.py --config configs/lca_clusters.yaml --model-type both --data-dir {shlex.quote(str(data_dir))}")

In [ ]:
# Tabla resumen del diagnóstico
import json
import pandas as pd

with open('results/diagnostico_mbe.json') as f:
    diag = json.load(f)

rows = []
for r in diag:
    row = {
        'clúster': r['cluster_id'],
        'mean(y)_train': round(r['y_mean_train'], 2),
        'mean(y)_test': round(r['y_mean_test'], 2),
        'Δmean(y)': round(r['delta_split_test_minus_train'], 2),
    }
    for mt in ('xgb', 'mlp'):
        if mt in r['models']:
            row[f'MBE_{mt}'] = round(r['models'][mt]['mbe'], 2)
            row[f'verdicto_{mt}'] = r['models'][mt]['veredicto']
    rows.append(row)
pd.DataFrame(rows)